# Preparação dos Dados

- **Projeto:** Recuperação da Confiabilidade dos Dados de Vendas
- **Etapa:** Pré-processamento dos dados
- **Dataset:** [Cafe Sales - Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training)
- **Notebook anterior:** [`01_initial-exploration.ipynb`](01_initial-exploration.ipynb)

Este notebook realiza a limpeza da base bruta de vendas, criação de novas variáveis para análises e exportação da base de dados limpas em Parquet.

**Sumário**

1. Carga da base bruta
2. Padronização de valores ausentes
3. Conversão de tipos
4. Duplicatas
5. Tratamento de valores ausentes
6. Validação das colunas tratadas
7. Feature Engineering
8. Salvando a base de dados limpa

In [1]:
import pandas as pd
import numpy as np

from src import load_raw_sales, calculate_df, dirty_mask

## 1. Carga da base bruta

A função `load_raw_sales` lê o CSV bruto e padroniza os nomes das colunas, da mesma forma que no notebook 01.

In [2]:
df = load_raw_sales('dirty_cafe_sales')
df.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 2. Padronização de valores ausentes

### 2.1 `ERROR` e `UNKNOWN` para `NaN`

`ERROR`, `UNKNOWN` e `NaN` representam o mesmo problema: valor ausente ou inválido. Unificar tudo em `NaN` facilita o tratamento desses valores ausentes.

> Foi usado o `mask` para que a regra do que é sujeira fique somente em `dirty_mask`, sem repetir a lista de valores inválidos neste notebook.

In [3]:
dirty_values = dirty_mask(df)
df = df.mask(dirty_values)

### 2.2 Validação

A verificação confirma duas coisas ao mesmo tempo: as únicas células `NaN` são as que estavam sujas antes, e nenhuma célula suja ficou de fora. Se alguma conta não bater, o `assert` interrompe a execução.

> células = colunas x linhas

In [4]:
assert (df.isna() == dirty_values).all().all()
print(f'Células padronizadas em NaN: {dirty_values.sum().sum()}')

Células padronizadas em NaN: 10082


### 2.3 Ausentes por coluna

Total e percentual de `NaN` por coluna, sobre as 10.000 linhas. Os valores batem perfeitamente com a tabela de sujeira total de [`01_initial-exploration.ipynb`](01_initial-exploration.ipynb).

In [5]:
calculate_df(df=df, column_name='total', aggregation=(df.isna()).sum())

,total,%
location,3961,39.61
payment_method,3178,31.78
item,969,9.69
price_per_unit,533,5.33
total_spent,502,5.02
quantity,479,4.79
transaction_date,460,4.60
transaction_id,0,0.00


## 3. Conversão de tipos

### 3.1 Tipos esperados

| Colunas | Tipo | Motivação |
|:--|:--|:--|
| `transaction_id` | `str` | Identificador: não entra em cálculos. |
| `item`, `payment_method`, `location` | `category` | Poucos valores que se repetem. |
| `price_per_unit`, `total_spent` | `float64` | Valores monetários com casas decimais. |
| `quantity` | `Int64` | Inteiro com suporte a `NaN`. |
| `transaction_date` | `datetime64` | Essencial para análises temporais. O formato é `AAAA-MM-DD`. |

### 3.2 Conversão

O `assert` compara a contagem de `NaN` antes e depois.

In [6]:
nan_before = df.isna().sum().copy()

In [7]:
data_types = {
    'item': 'category', 'quantity': 'Int64', 'price_per_unit': 'float64',
    'total_spent': 'float64', 'payment_method': 'category', 'location': 'category'
}

df = df.astype(data_types)

df['transaction_date'] = pd.to_datetime(df['transaction_date'], format='%Y-%m-%d')

In [8]:
assert (df.isna().sum() == nan_before).all()

df.dtypes

transaction_id                 str
item                      category
quantity                     Int64
price_per_unit             float64
total_spent                float64
payment_method            category
location                  category
transaction_date    datetime64[us]
dtype: object

### 3.3 Resumo estatístico 

In [9]:
df.describe()

,quantity,price_per_unit,total_spent,transaction_date
count,9521.0,9467.000000,9498.000000,9540
mean,3.028463,2.949984,8.924352,2023-07-01 23:00:31.698113
min,1.0,1.000000,1.000000,2023-01-01 00:00:00
25%,2.0,2.000000,4.000000,2023-04-01 00:00:00
50%,3.0,3.000000,8.000000,2023-07-02 00:00:00
75%,4.0,4.000000,12.000000,2023-10-02 00:00:00
max,5.0,5.000000,25.000000,2023-12-31 00:00:00
std,1.419007,1.278450,6.009919,NaN


## 4. Duplicatas

### 4.1 Linhas repetidas sem o `transaction_id`

O `transaction_id` é único, então cada linha é uma venda distinta. A verificação abaixo desconsidera o ID para ver se existem vendas com todos os demais atributos iguais.

In [10]:
line_duplicated = df.duplicated(subset=[col for col in df.columns if col != 'transaction_id']).sum()

print(f'Linhas repetidas nas demais colunas: {line_duplicated}')

Linhas repetidas nas demais colunas: 255


**Conclusão:** 255 linhas repetem outra nas demais colunas. Como os IDs são únicos, são vendas diferentes que coincidem nos atributos, o que é esperado numa base com poucas categorias. Nenhuma linha é removida.

## 5. Tratamento de valores ausentes

### 5.1 Relação entre quantidade, preço e total

Antes de usar a relação `quantity × price_per_unit = total_spent` para recuperar valores, é preciso confirmar que ela vale nas linhas em que as três colunas estão preenchidas. Caso a diferença entre as linhas válidas e divergentes seja maior ou igual a `0.01`, quer dizer que a regra não pode ser imputada.

In [11]:
valids = df.dropna(subset=['quantity', 'price_per_unit', 'total_spent'])

divergents = ((valids['quantity'] * valids['price_per_unit'] - valids['total_spent']).abs() > 0.01).sum()

print(f"Total de linhas com as 3 colunas válidas: {len(valids)}")
print(f"Linhas com divergência: {divergents}")

assert divergents == 0, f"A relação falha em {divergents} linhas"

Total de linhas com as 3 colunas válidas: 8544
Linhas com divergência: 0


**Conclusão:** a relação vale nas 8.544 linhas com as três colunas preenchidas, sem nenhuma divergência. Posso usá-la para recuperar valores ausentes.

### 5.2 Recuperação do preço a partir do item

Cada item é vendido por um preço fixo. Primeiro confirmo que cada item tem um único preço nos dados e monto a tabela item → preço. Depois uso essa tabela para preencher o `price_per_unit` ausente nas linhas em que o item é conhecido.

| `item` | `price` |
|:-|--|
| Cookie | $1.00 |
| Tea | $1.50 |
| Coffee | $2.00 |
| Cake ou Juice | $3.00 | 
| Sandwich ou Smoothie | $4.00 | 
| Salad | $5.00 |

Dois itens estão relacionados a dois preços, Cake e Juice vendidos a $3.00, Sandwich e Smoothie a $4.00. Portanto, não é possível afirmar qual a melhor substituição do preço usando esses itens.

In [12]:
price_nan = df['price_per_unit'].isna().sum()

known = df.dropna(subset=['item', 'price_per_unit'])
assert (known.groupby('item', observed=True)['price_per_unit'].nunique() == 1).all()
price_by_item = known.groupby('item', observed=True)['price_per_unit'].first().to_dict()

df['price_per_unit'] = df['price_per_unit'].fillna(df['item'].astype(object).map(price_by_item))

print(f"Preços ausentes antes: {price_nan}")
print(f"Preços ausentes depois: {df['price_per_unit'].isna().sum()}")

Preços ausentes antes: 533
Preços ausentes depois: 54


**Conclusão:** o agrupamento recuperou 479 preços ausentes. Restam 54 linhas sem preço, que podem ser tratadas com a divisão entre o preço total e a quantidade.

### 5.3 Recuperação do preço a partir do total e da quantidade

A relação `quantity × price_per_unit = total_spent` foi validada na 5.1. Posso usá-la para calcular o `price_per_unit` ausente nas linhas em que `total_spent` e `quantity` estão preenchidos, dividindo o total pela quantidade.

In [13]:
price_nan = df['price_per_unit'].isna().sum()
df['price_per_unit'] = df['price_per_unit'].fillna((df['total_spent'] / df['quantity']).round(2))

print(f"Preços ausentes antes: {price_nan}")
print(f"Preços ausentes depois: {df['price_per_unit'].isna().sum()}")

Preços ausentes antes: 54
Preços ausentes depois: 6


**Conclusão:** a divisão recuperou 48 preços ausentes. Restam 6 linhas sem preço, em que também falta o total ou a quantidade.

### 5.4 Recuperação do item a partir do preço

Cada preço pode corresponder a mais de um item. Só é possível recuperar o item quando o preço aponta para um único item. Os preços $3.00 e $4.00 tem 2 produtos relacionados, com isso não é possível afirmar com certeza qual item está ligado a esses preços. Nesse caso, decidi por transformar esses itens em uma categoria de "Not Specified". 

In [14]:
# Usa a variável known de 5.2
item_nan = df['item'].isna().sum()

qty_items_price = known.groupby('price_per_unit', observed=True)['item'].nunique()
item_by_price = (known[known['price_per_unit'].map(qty_items_price) == 1]
                                              .groupby('price_per_unit', observed=True)['item']
                                              .first().to_dict())

df['item'] = (df['item'].cat.add_categories('Not Specified')
                            .fillna(df['price_per_unit'].map(item_by_price))
                            .fillna('Not Specified'))

print(f"Itens ausentes antes: {item_nan}")
print(f"Itens ausentes depois: {df['item'].isna().sum()}")
print(f"Itens como 'Not Specified': {(df['item'] == 'Not Specified').sum()}")

Itens ausentes antes: 969
Itens ausentes depois: 0
Itens como 'Not Specified': 480


**Conclusão:** o preço recuperou 489 itens. Os outros 480, que têm preço ambíguo (3.0 ou 4.0) ou não têm preço, foram marcados como "Not Specified". A coluna não tem mais valores ausentes.

### 5.5 Recuperação da quantidade a partir do total e do preço

Calculo a `quantity` ausente dividindo `total_spent` por `price_per_unit` nas linhas em que os dois estão preenchidos. Como a quantidade é um número inteiro, arredondo o resultado.

In [15]:
quantity_nan = df['quantity'].isna().sum()
df['quantity'] = df['quantity'].fillna((df['total_spent'] / df['price_per_unit']).round().astype('Int64'))

print(f"Quantidades ausentes antes: {quantity_nan}")
print(f"Quantidades ausentes depois: {df['quantity'].isna().sum()}")

Quantidades ausentes antes: 479
Quantidades ausentes depois: 23


**Conclusão:** a divisão recuperou 456 quantidades ausentes. Restam 23 linhas sem quantidade, em que também falta o total ou o preço unitário.

### 5.6 Recuperação do total a partir da quantidade e do preço

Calculo o `total_spent` ausente multiplicando `quantity` por `price_per_unit` nas linhas em que os dois estão preenchidos.

In [16]:
total_spent_nan = df['total_spent'].isna().sum()
df['total_spent'] = df['total_spent'].fillna(df['quantity'] * df['price_per_unit'])

print(f"Preços totais ausentes antes: {total_spent_nan}")
print(f"Preços totais ausentes depois: {df['total_spent'].isna().sum()}")

Preços totais ausentes antes: 502
Preços totais ausentes depois: 23


**Conclusão:** a multiplicação recuperou 479 preços totais ausentes. Restam 23 linhas sem preço total, em que também falta o preço unitário ou a quantidade.

### 5.7 Tratamento de `payment_method`

A coluna `payment_method` tem 3.178 valores ausentes, cerca de 31.8% da base. Para decidir o tratamento, primeiro olho a proporção de cada valor entre as linhas preenchidas.

In [17]:
payment_method_filled = df[['payment_method']].dropna()
calculate_df(df=payment_method_filled, column_name='total', aggregation=payment_method_filled.value_counts()).reset_index()

,payment_method,total,%
0,Digital Wallet,2291,33.58
1,Credit Card,2273,33.32
2,Cash,2258,33.10


Os três métodos de pagamentos têm proporções quase iguais, cerca de 33% cada, com o mais frequente superando os outros por apenas 18 linhas. Preencher os NaN com a moda vai distorcer os dados. Por isso optei por transformar os valores ausentes na categória "Not Specified".

In [18]:
payment_method_nan = df['payment_method'].isna().sum()

df['payment_method'] = (df['payment_method'].cat.add_categories('Not Specified')
                                                .fillna('Not Specified'))

print(f"Métodos de pagamentos ausentes antes: {payment_method_nan}")
print(f"Métodos de pagamentos ausentes depois: {df['payment_method'].isna().sum()}")
print(f"Métodos de pagamentos como 'Not Specified': {(df['payment_method'] == 'Not Specified').sum()}")

Métodos de pagamentos ausentes antes: 3178
Métodos de pagamentos ausentes depois: 0
Métodos de pagamentos como 'Not Specified': 3178


**Conclusão:** nenhum valor de `payment_method` foi recuperado. As 3.178 linhas ausentes foram marcadas como "Not Specified", e a coluna não tem mais valores ausentes.

### 5.8 Tratamento de `location`

A coluna `location` tem 3.961 valores ausentes, cerca de 39.6% da base. Para decidir o tratamento, primeiro olho a proporção de cada valor entre as linhas preenchidas.

In [19]:
location_filled = df[['location']].dropna()
calculate_df(df=location_filled, column_name='total', aggregation=location_filled.value_counts()).reset_index()

,location,total,%
0,Takeaway,3022,50.04
1,In-store,3017,49.96


Os valores de `location` também têm proporções quase iguais, cerca de 50% cada, e o mais frequente supera o outro por apenas 5 linhas. Assim como na coluna `payment_method`, preencher os valores com a moda também distorcerá os dados. Optei por transformar os valores ausentes na categória "Not Specified".

In [20]:
location_nan = df['location'].isna().sum()

df['location'] = (df['location'].cat.add_categories('Not Specified')
                                    .fillna('Not Specified'))

print(f"Localização ausentes antes: {location_nan}")
print(f"Localização ausentes depois: {df['location'].isna().sum()}")
print(f"Localização como 'Not Specified': {(df['location'] == 'Not Specified').sum()}")                                    

Localização ausentes antes: 3961
Localização ausentes depois: 0
Localização como 'Not Specified': 3961


**Conclusão:** nenhum valor de `location` foi recuperado. As 3.961 linhas ausentes foram marcadas como "Not Specified", e a coluna não tem mais valores ausentes.

### 5.9 Descarte das linhas numéricas sem solução

A relação `quantity × price_per_unit = total_spent` resolveu uma grande parte das linhas ausentes. Mas não foi possível recuperar quando algumas das colunas apresentavam valores NaN, por isso essas linhas serão descartadas.

In [21]:
rows_before = len(df)

df = df.dropna(subset=['quantity', 'price_per_unit', 'total_spent']).reset_index(drop=True)

print(f"Linhas removidas: {rows_before - len(df)}")
print(f"Linhas restantes: {len(df)}")

Linhas removidas: 26
Linhas restantes: 9974


**Conclusão:** foram removidas 26 linhas, cerca de 0.26% da base de dados, todas com duas ou mais colunas numéricas ausentes. As três colunas numéricas ficaram completas.

### 5.10 Tratamento de `transaction_date`

A coluna de `transaction_date` tem 460 linhas com valores ausentes e descartar essas linhas significaria perder cerca de 4.6% do faturamento real de vendas, visto que os nulos da coluna `total_spent` já foram descartados.

In [22]:
(df['total_spent'].isna() & df['transaction_date'].isna()).sum()

np.int64(0)

Outra opção para tratamento para as linhas ausentes é imputar pela data anterior e posterior, mas deve considerar que os valores de `transaction_id` foram gerados de forma sequencial. 

In [23]:
df[['transaction_id', 'transaction_date']].dropna().sort_values('transaction_id', ascending=False).head()

,transaction_id,transaction_date
98,TXN_9999113,2023-05-27
4291,TXN_9997620,2023-12-11
7947,TXN_9997263,2023-03-05
1385,TXN_9996968,2023-10-14
2490,TXN_9996195,2023-03-16


Como pode ser observado, não é possível definir uma ordem para a data das vendas. Sendo assim, não é viável usar a regra abordada de usar tratar com a data anterior ou posterior.

Outra análise importante é comparar o valor médio das vendas com e sem as data ausentes, para ver se as linhas sem data se comportam de forma diferente das demais.

In [24]:
no_date = df['transaction_date'].isna()
df.groupby(no_date)['total_spent'].agg(['count', 'mean']).reset_index()

,transaction_date,count,mean
0,False,9514,8.92038
1,True,460,9.072826


O total médio é 9.07 nas vendas sem data e 8.92 nas vendas com data. A diferença é pequena, e não há sinal de que as linhas sem data sejam diferentes das demais.

**Conclusão:** nenhum valor de `transaction_date` foi recuperado ou imputado. As 460 linhas sem data foram mantidas com `NaT`, isso representa cerca de 4.61% da base de dados. 

### 5.11 Resumo estatístico depois do tratamento

Quantidade, preço e total vêm de um cardápio com preços fixos, então o que conta como discrepante é definido pelo cardápio e não pela distribuição. Confiro os intervalos das colunas numéricas, os preços distintos e o período das datas.

In [25]:
df.describe()

,quantity,price_per_unit,total_spent,transaction_date
count,9974.0,9974.0,9974.0,9514
mean,3.024865,2.946962,8.927411,2023-07-01 23:14:35.593861
min,1.0,1.0,1.0,2023-01-01 00:00:00
25%,2.0,2.0,4.0,2023-04-01 00:00:00
50%,3.0,3.0,8.0,2023-07-02 00:00:00
75%,4.0,4.0,12.0,2023-10-02 00:00:00
max,5.0,5.0,25.0,2023-12-31 00:00:00
std,1.420504,1.280144,6.002644,NaN


**Conclusão:** Os valores estão dentro do esperado, não é necessário nenhum tratamento.

### 5.12 Resumo do tratamento de valores ausentes

| Coluna | Tratamento |
|:-|:-|
| `item` | Recuperado pelo preço quando o preço aponta para um único item; o restante recebeu "Not Specified" |
| `quantity` | Recuperada por `total_spent / price_per_unit`; linhas sem solução descartadas |
| `price_per_unit` | Recuperado pelo item e por `total_spent ÷ quantity`; linhas sem solução descartadas |
| `total_spent` | Recuperado por `quantity × price_per_unit`; linhas sem solução descartadas |
| `payment_method` | Nenhum valor recuperado; ausentes receberam "Not Specified" |
| `location` | Nenhum valor recuperado; ausentes receberam "Not Specified" |
| `transaction_date` | Nenhum valor recuperado; ausentes mantidos como `NaT` |

**Conclusão:** a base final tem 9.974 linhas.  O que não pôde ser recuperado por regra foi marcado como "Not Specified" ou mantido ausente, como na coluna `transaction_date`.

## 6. Validação das colunas tratadas

Antes de seguir para o feature engineering, confirmo que a base final cumpre o que a limpeza prometeu: o único ausente é `transaction_date`, os IDs são únicos e a relação `quantity × price_per_unit = total_spent` vale em todas as linhas.

In [26]:
assert set(df.columns[df.isna().any()]) == {'transaction_date'}, 'Há ausentes fora de transaction_date'
assert df['transaction_id'].is_unique, 'Há IDs repetidos'
assert ((df['quantity'] * df['price_per_unit'] - df['total_spent']).abs() <= 0.01).all(), 'A relação falha em alguma linha'

print(f'Linhas finais: {len(df)}')

Linhas finais: 9974


**Conclusão:** a base final tem 9.974 linhas. A única coluna com NaN é `transaction_date`, mantido de propósito. Os IDs são únicos e a relação entre quantidade, preço e total vale em todas as linhas.

## 7. Feature Engineering

As perguntas de negócio por período partem de `transaction_date`, é importante conseguir calcular o trimestre, mês e dia da semana. Crio três colunas a partir dela: `month`, `quarter` e `weekday` para análises sazonais.

In [27]:
df['month'] = df['transaction_date'].dt.month.astype('Int64')
df['quarter'] = df['transaction_date'].dt.quarter.astype('Int64')
df['weekday'] = df['transaction_date'].dt.day_name()

In [28]:
df[['transaction_date', 'month', 'quarter', 'weekday']].head()

,transaction_date,month,quarter,weekday
0,2023-09-08,9,3,Friday
1,2023-05-16,5,2,Tuesday
2,2023-07-19,7,3,Wednesday
3,2023-04-27,4,2,Thursday
4,2023-06-11,6,2,Sunday


## 8. Salvando a base de dados limpa

> Optei por salvar a base de dados limpas em Parquet para garantir a integridade dos dados.

In [29]:
df.to_parquet(r'../data/processed/clean_cafe_sales.parquet', index=False)

**Conclusão:** A base final tem 9.974 linhas, com 11 colunas. A preparação removeu os valores inválidos, recuperou dados por regra aritmética, manteve a integridade do faturamento e criou variáveis temporais.